## Overview 
 data analyst at the fictional Mint Classics Company, helping to analyze data in a relational database with the goal of supporting inventory-related business decisions that lead to the closure of a storage facility.

## Project Scenario
a retailer of classic model cars and other vehicles, is looking at closing one of their storage facilities. 
To support a data-based business decision, they are looking for suggestions and recommendations for reorganizing or reducing inventory, while still maintaining timely service to their customers

#### Step 1: Import our data.


In [1]:
# Importar librerías
import pymysql
import pandas as pd
import numpy as np
from sqlalchemy import create_engine


In [2]:
from dotenv import load_dotenv
import os

load_dotenv() 

# Obtener las credenciales
DB_HOST = os.getenv('DB_HOST')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_NAME = os.getenv('DB_NAME')

In [3]:
conn = pymysql.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)

print("Conectado a mintclassics")
cursor = conn.cursor()
cursor.execute("SHOW TABLES")

for table in cursor.fetchall():
    print(table)

Conectado a mintclassics
('customers',)
('employees',)
('offices',)
('orderdetails',)
('orders',)
('payments',)
('productlines',)
('products',)
('warehouses',)


In [4]:
df = pd.read_sql(
    "SELECT * FROM products LIMIT 2",
    conn
)

df.head(2)

C:\Users\DELL\AppData\Local\Temp\ipykernel_11864\1183747223.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,warehouseCode,buyPrice,MSRP
0,S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front...",7933,a,48.81,95.7
1,S10_1949,1952 Alpine Renault 1300,Classic Cars,1:10,Classic Metal Creations,Turnable front wheels; steering function; deta...,7305,b,98.58,214.3


#### Step 2: EDA and Clean the data.


In [5]:
# Display current schema
tables = ['customers','employees','offices','orderdetails','orders','payments','productlines','products','warehouses']

for table in tables:
    print(f"Table: {table}")
    cursor = conn.cursor()
    cursor.execute(f"DESCRIBE {table}") 
    for column in cursor.fetchall():
        print(f"  {column[0]} ({column[1]})")
    print()


Table: customers
  customerNumber (int)
  customerName (varchar(50))
  contactLastName (varchar(50))
  contactFirstName (varchar(50))
  phone (varchar(50))
  addressLine1 (varchar(50))
  addressLine2 (varchar(50))
  city (varchar(50))
  state (varchar(50))
  postalCode (varchar(15))
  country (varchar(50))
  salesRepEmployeeNumber (int)
  creditLimit (decimal(10,2))

Table: employees
  employeeNumber (int)
  lastName (varchar(50))
  firstName (varchar(50))
  extension (varchar(10))
  email (varchar(100))
  officeCode (varchar(10))
  reportsTo (int)
  jobTitle (varchar(50))

Table: offices
  officeCode (varchar(10))
  city (varchar(50))
  phone (varchar(50))
  addressLine1 (varchar(50))
  addressLine2 (varchar(50))
  state (varchar(50))
  country (varchar(50))
  postalCode (varchar(15))
  territory (varchar(10))

Table: orderdetails
  orderNumber (int)
  productCode (varchar(15))
  quantityOrdered (int)
  priceEach (decimal(10,2))
  orderLineNumber (smallint)

Table: orders
  orderNumbe

I give the last output to a Generative-AI to search a possible incorrect type, the result is possible error in Warehouses, so i going to investigate by myself

In [6]:
df = pd.read_sql(
    "SELECT * FROM warehouses LIMIT 5",
    conn
)
df.head()
#Conclusion: it is correct, no error in the type

C:\Users\DELL\AppData\Local\Temp\ipykernel_11864\3664919713.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,warehouseCode,warehouseName,warehousePctCap
0,a,North,72
1,b,East,67
2,c,West,50
3,d,South,75


In [7]:
query = """
SELECT COUNT(*) AS products_without_warehouse
FROM products
WHERE warehouseCode IS NULL;
"""
pd.read_sql(query, conn)


C:\Users\DELL\AppData\Local\Temp\ipykernel_11864\3044841269.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query, conn)


,products_without_warehouse
0,0


In [8]:
query = """
SELECT productCode, productName, quantityInStock
FROM products
WHERE quantityInStock < 0
   OR quantityInStock IS NULL;
"""

df = pd.read_sql(query, conn)
df

C:\Users\DELL\AppData\Local\Temp\ipykernel_11864\397263439.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,productCode,productName,quantityInStock


In [9]:
query = """
SELECT productCode,
       COUNT(*) AS duplicados
FROM products
GROUP BY productCode
HAVING COUNT(*) > 1;
"""

df = pd.read_sql(query, conn)
df

C:\Users\DELL\AppData\Local\Temp\ipykernel_11864\319148429.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,productCode,duplicados


In [10]:
query = """
SELECT *
FROM orders
"""

df = pd.read_sql(query, conn)

df.isnull().sum()

C:\Users\DELL\AppData\Local\Temp\ipykernel_11864\2352695404.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


orderNumber         0
orderDate           0
requiredDate        0
shippedDate        14
status              0
comments          246
customerNumber      0
dtype: int64

In [11]:
engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")
query = """
SELECT *
FROM orders
WHERE shippedDate IS NULL AND status <> 'Cancelled'
"""

df = pd.read_sql(query, engine)

display(df)

,orderNumber,orderDate,requiredDate,shippedDate,status,comments,customerNumber
0,10334,2004-11-19,2004-11-28,None,On Hold,The outstaniding balance for this customer exc...,144
1,10401,2005-04-03,2005-04-14,None,On Hold,Customer credit limit exceeded. Will ship when...,328
2,10407,2005-04-22,2005-05-04,None,On Hold,Customer credit limit exceeded. Will ship when...,450
3,10414,2005-05-06,2005-05-13,None,On Hold,Customer credit limit exceeded. Will ship when...,362
4,10420,2005-05-29,2005-06-07,None,In Process,None,282
5,10421,2005-05-29,2005-06-06,None,In Process,Custom shipping instructions were sent to ware...,124
6,10422,2005-05-30,2005-06-11,None,In Process,None,157
7,10423,2005-05-30,2005-06-05,None,In Process,None,314
8,10424,2005-05-31,2005-06-08,None,In Process,None,141
9,10425,2005-05-31,2005-06-07,None,In Process,None,119


The database looks clean

## Step 3: Business/Data Analysis

In [12]:
#Inventory by warehouse
query ="""
SELECT warehouseCode,COUNT(*) AS products_total,SUM(quantityInStock) AS total_inventory
FROM products
GROUP BY warehouseCode
ORDER BY total_inventory;
"""
df = pd.read_sql(query, conn)
df
#df.to_csv("inventory_by_warehouse.csv", index=False)

C:\Users\DELL\AppData\Local\Temp\ipykernel_11864\1727544595.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,warehouseCode,products_total,total_inventory
0,d,23,79380.0
1,c,24,124880.0
2,a,25,131688.0
3,b,38,219183.0


In [13]:
#Sales by warehouse
query ="""
SELECT p.warehouseCode,SUM(od.quantityOrdered) AS sold_units
FROM orderdetails od
JOIN products p
    ON od.productCode = p.productCode
GROUP BY p.warehouseCode
ORDER BY sold_units;
"""
df = pd.read_sql(query, conn)
df
#df.to_csv("Sales by warehouse.csv", index=False)

C:\Users\DELL\AppData\Local\Temp\ipykernel_11864\98465545.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,warehouseCode,sold_units
0,d,22351.0
1,c,22933.0
2,a,24650.0
3,b,35582.0


In [14]:
#Products never sold
query ="""
SELECT
    p.productCode,
    p.productName,
    p.warehouseCode,
    p.quantityInStock
FROM products p
LEFT JOIN orderdetails od
    ON p.productCode = od.productCode
WHERE od.productCode IS NULL;
"""
df = pd.read_sql(query, conn)
df
#df.to_csv("Products never sold.csv", index=False)

C:\Users\DELL\AppData\Local\Temp\ipykernel_11864\705524787.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,productCode,productName,warehouseCode,quantityInStock
0,S18_3233,1985 Toyota Supra,b,7733


In [15]:
#Inventory vs. Demand
query ="""
SELECT
    p.productCode,
    p.productName,
    p.warehouseCode,
    p.quantityInStock,
    COALESCE(SUM(od.quantityOrdered),0) AS Sold_total
FROM products p
LEFT JOIN orderdetails od
    ON p.productCode = od.productCode
GROUP BY p.productCode,
         p.productName,
         p.warehouseCode,
         p.quantityInStock
ORDER BY Sold_total
LIMIT 10;
"""
df = pd.read_sql(query, conn)
df
#df.to_csv("Inventory vs. Demand.csv", index=False)

C:\Users\DELL\AppData\Local\Temp\ipykernel_11864\1983569722.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,productCode,productName,warehouseCode,quantityInStock,Sold_total
0,S18_3233,1985 Toyota Supra,b,7733,0.0
1,S18_4933,1957 Ford Thunderbird,b,3209,767.0
2,S24_1046,1970 Chevy Chevelle SS 454,b,1005,803.0
3,S24_3969,1936 Mercedes Benz 500k Roadster,c,2081,824.0
4,S18_2248,1911 Ford Town Car,c,540,832.0
5,S18_2870,1999 Indy 500 Monte Carlo SS,b,8164,855.0
6,S18_4409,1932 Alfa Romeo 8C2300 Spider Sport,c,6553,866.0
7,S24_4048,1992 Porsche Cayenne Turbo Silver,b,6582,867.0
8,S24_3191,1969 Chevrolet Camaro Z28,b,4695,870.0
9,S24_2887,1952 Citroen-15CV,b,1452,873.0


In [16]:
#Value inventory
query ="""
SELECT
    warehouseCode,
    SUM(quantityInStock * buyPrice) AS inventory_value
FROM products
GROUP BY warehouseCode
ORDER BY inventory_value;
"""
df = pd.read_sql(query, conn)
df
#df.to_csv("Value inventory.csv", index=False)

C:\Users\DELL\AppData\Local\Temp\ipykernel_11864\1538581392.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,warehouseCode,inventory_value
0,d,4105721.76
1,c,5704259.82
2,a,6664996.94
3,b,14059337.71


In [17]:
#warehousePctCap
query ="""SELECT * FROM warehouses ORDER BY warehousePctCap DESC """
df = pd.read_sql(query, conn)
df
#df.to_csv("warehousePctCap.csv", index=False)

C:\Users\DELL\AppData\Local\Temp\ipykernel_11864\3139342256.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,warehouseCode,warehouseName,warehousePctCap
0,d,South,75
1,a,North,72
2,b,East,67
3,c,West,50


## Conclusion

After analyzing inventory levels, sales activity, inventory value, product demand, and warehouse utilization, Warehouse D appears to be the strongest candidate for closure. It has the lowest inventory volume, the lowest sales contribution, and the lowest inventory value among all warehouses. These findings suggest that closing Warehouse D would likely have the smallest impact on customer service and overall business operations.

Furthermore, the remaining warehouses have sufficient unused capacity to absorb Warehouse D's inventory without significantly affecting service levels. This indicates that inventory redistribution can be achieved while maintaining operational efficiency and timely customer deliveries.

Additionally, the analysis of product demand revealed several slow-moving products in Warehouse B, including at least one product with no recorded sales. While Warehouse B is not a strong candidate for closure due to its high inventory value, these findings highlight an opportunity to optimize inventory management and improve stock turnover. Reducing excess inventory and reassessing low-demand products could further enhance operational efficiency and free up storage capacity.

Therefore, the recommended strategy is to close Warehouse D and redistribute its inventory across the remaining warehouses. At the same time, Mint Classics should implement inventory optimization initiatives, particularly in Warehouse B, to reduce excess stock, improve inventory turnover, and maximize warehouse utilization.

### Tableau Dashboard

https://public.tableau.com/app/profile/hector.salas/viz/MintClassicsInventoryWarehousesOptimizationAnalysis/MintClassicsInventoryOptimizationAnalysis

In [18]:
cursor.close()   
conn.close()     
engine.dispose()   